# Bucketing

**What this notebook does**

* Joins `orders` (1,000 rows) with `products` (100 rows) on `product_id`, with and without bucketing.
* Compares physical plans for a join, an aggregation and a filter (bucket pruning).

**Core concept: bucketing**

`bucketBy(n, col)` hashes the column into `n` buckets when writing a table, so rows with the same key always land in the same bucket file. If two tables are bucketed the same way on the join key, Spark can join or aggregate them bucket-by-bucket **without a shuffle** (no `Exchange` in the plan). A filter on the bucket column can also read only the matching bucket (**bucket pruning**).

**Databricks conversion notes**

* Original local Jupyter session setup (`SparkSession.builder`, `sparkContext`, `spark.driver.memory`) removed: Databricks provides `spark`, and driver memory is set by the compute.
* CSV paths now point to the Unity Catalog volume `/Volumes/lingaro_sandbox_ne/sandani_sandbox/volume_managed`.
* Tables are written to `lingaro_sandbox_ne.sandani_sandbox` instead of the local `spark-warehouse`.
* **Unity Catalog managed tables are Delta tables, and Delta does not support `bucketBy`.** The notebook tries bucketing first; if it fails it falls back to the Databricks-recommended equivalent, **liquid clustering** (`CLUSTER BY product_id`). Liquid clustering gives data skipping (similar to bucket pruning) but does not remove the join/aggregation shuffle.
* On serverless, `spark.sql.autoBroadcastJoinThreshold` cannot be changed, so the 100-row products table is broadcast and you will see `BroadcastHashJoin` instead of `SortMergeJoin`. Use a classic cluster (e.g. Sandani Personal Compute) to reproduce the original plans.

## 1. Setup

In [0]:
import warnings
warnings.filterwarnings("ignore")

In [0]:
# Jupyter-only InteractiveShell setting removed; Databricks renders cell output natively.

# Source files (Unity Catalog volume) and target schema for the tables we write
base_path = "/Volumes/lingaro_sandbox_ne/sandani_sandbox/volume_managed"
target_schema = "lingaro_sandbox_ne.sandani_sandbox"

In [0]:
from pyspark.storagelevel import StorageLevel
from pyspark.sql.types import *
import pyspark.sql.functions as F
# SparkSession is not imported: Databricks provides a ready-to-use `spark` object.

In [0]:
# Original (local Jupyter):
#   spark = SparkSession.builder.config("spark.driver.memory", "10g").master("local[*]").getOrCreate()
#   sc = spark.sparkContext; sc.setLogLevel("ERROR")
# On Databricks the session already exists; sparkContext is not available under Spark Connect.

# Disable broadcast joins so the small products table does not hide the shuffle
try:
    spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)
    print("Broadcast joins disabled")
except Exception as e:
    print(f"Could not change autoBroadcastJoinThreshold on this compute (serverless): {type(e).__name__}")

In [0]:
orders_file = f"{base_path}/orders.csv"
df_orders = spark.read.csv(orders_file, header=True, inferSchema=True)

In [0]:
display(df_orders.limit(5))
df_orders.printSchema()

In [0]:
products_file = f"{base_path}/products.csv"
df_products = spark.read.csv(products_file, header=True, inferSchema=True)

In [0]:
display(df_products.limit(5))
df_products.printSchema()

In [0]:
df_products.select("product_id").distinct().count()

In [0]:
df_orders.select("order_id").distinct().count()

## 2. Bucketing In Joins

**Baseline (no bucketing):** join the two DataFrames read from CSV.

**Core concept:** With broadcast disabled, Spark uses a **sort-merge join**: both sides are shuffled by `product_id` (`Exchange hashpartitioning(product_id, ...)` in the plan) and sorted before merging. The shuffle is the expensive part that bucketing aims to remove. `explain()` prints the physical plan; `count()` is the action that actually runs the join.

In [0]:
df_orders_product_details = (
    df_orders.join(
        df_products,
        on="product_id",
        how="inner"
    )
)

In [0]:
df_orders_product_details.explain()

In [0]:
df_orders_product_details.count()

**What:** Write both tables bucketed into 4 buckets on `product_id`, read them back and repeat the join.

**Core concept:** Bucketing only helps when both sides use the **same column and the same number of buckets**; then Spark knows bucket *i* of orders only matches bucket *i* of products, so the plan has no `Exchange` before the `SortMergeJoin`. Bucket metadata is stored in the table catalog, which is why it must be written with `saveAsTable` (not plain `write.parquet`).

**On Unity Catalog:** `bucketBy` fails because managed tables are Delta. The fallback writes `orders_clustered` and `products_clustered` with `CLUSTER BY (product_id)`. Expect the join plan to still contain `Exchange` (or a broadcast on serverless); the benefit of clustering shows up in filters (section 4), not in removing shuffles.

In [0]:
# Original: df_products.write.bucketBy(4, col="product_id").mode("overwrite").saveAsTable("products_bucketed")
# Unity Catalog managed tables are Delta, which does not support bucketBy, so failures are caught here.
NUM_BUCKETS = 4
bucketing_supported = True
try:
    (
        df_products
        .write.bucketBy(NUM_BUCKETS, col="product_id")
        .mode("overwrite")
        .saveAsTable(f"{target_schema}.products_bucketed")
    )
    print("products_bucketed created")
except Exception as e:
    bucketing_supported = False
    print(f"bucketBy not supported here -> falling back to liquid clustering.\n{str(e).splitlines()[0][:300]}")

In [0]:
if bucketing_supported:
    (
        df_orders
        .write.bucketBy(NUM_BUCKETS, col="product_id")
        .mode("overwrite")
        .saveAsTable(f"{target_schema}.orders_bucketed")
    )
    orders_table = f"{target_schema}.orders_bucketed"
    products_table = f"{target_schema}.products_bucketed"
else:
    # Databricks alternative: Delta tables with liquid clustering on the join key
    orders_table = f"{target_schema}.orders_clustered"
    products_table = f"{target_schema}.products_clustered"
    df_orders.createOrReplaceTempView("orders_src")
    df_products.createOrReplaceTempView("products_src")
    spark.sql(f"CREATE OR REPLACE TABLE {orders_table} CLUSTER BY (product_id) AS SELECT * FROM orders_src")
    spark.sql(f"CREATE OR REPLACE TABLE {products_table} CLUSTER BY (product_id) AS SELECT * FROM products_src")

print(f"orders table:   {orders_table}\nproducts table: {products_table}")

In [0]:
# Variable names kept from the original; they point to the clustered tables when bucketing is unavailable
df_orders_bucketed = spark.table(orders_table)
df_products_bucketed = spark.table(products_table)

In [0]:
df_orders_product_details_bucketed = (
    df_orders_bucketed.join(
        df_products_bucketed,
        on="product_id",
        how="inner"
    )
)

In [0]:
df_orders_product_details_bucketed.explain()

In [0]:
df_orders_product_details_bucketed.count()

## 3. Bucketing In Aggregations

**What:** Sum `total_amount` per `product_id`, first on the plain DataFrame, then on the bucketed table.

**Core concept:** A `groupBy` normally needs a shuffle so that all rows of a key meet in one partition (`HashAggregate -> Exchange -> HashAggregate`). If the table is bucketed on the grouping column, the rows are already co-located, so Spark can skip the `Exchange`. Liquid clustering does not give Spark this partitioning guarantee. Note: on the small clustered table here the `Exchange` may still disappear, but only because the 1,000-row table is read as a single partition. On larger, multi-file tables the shuffle remains.

In [0]:
display(df_orders.limit(5))

In [0]:
# WITHOUT BUCKETING

df_product_sales = (
    df_orders
    .groupBy("product_id")
    .agg(F.sum("total_amount").alias("sales"))
)

df_product_sales.explain()

In [0]:
# WITH BUCKETING (or liquid clustering on Unity Catalog)

df_product_sales = (
    df_orders_bucketed
    .groupBy("product_id")
    .agg(F.sum("total_amount").alias("sales"))
)

df_product_sales.explain()

## 4. Bucket Pruning

**What:** Filter on `product_id == 1` before aggregating.

**Core concept:** For a bucketed table Spark computes which bucket holds `product_id = 1` and reads only that bucket file; the plan's scan node shows `SelectedBucketsCount: 1 out of 4`. For a liquid-clustered Delta table the equivalent is **data skipping**: Delta uses per-file min/max statistics on the clustering column to skip files, visible in the plan as a pushed-down filter and in the query profile as files pruned.

In [0]:
df_product_sales_bucket_pruning = (
    df_orders_bucketed
    .filter(F.col("product_id") == 1)
    .groupBy("product_id")
    .agg(F.sum("total_amount").alias("sales"))
)

In [0]:
df_product_sales_bucket_pruning.explain()